# 기술 스택 추출 파이프라인

ATS 채용공고 데이터의 `description`에서 실제 요구되는 기술 스택을 뽑아 `skills` 열로 저장한다.
채용 데이터를 계속 새로 수집할 예정이라, 자동으로 매번 돌리는 **메인 파이프라인**과
필요할 때만 수동으로 돌리는 **부록**을 분리했다.

메인 파이프라인:

0. **중복 공고 제거** — description 완전 중복 + company/title 그룹 내 근접 중복(유사도 0.95↑) 제거
1. **TECH_DICTIONARY** — canonical 기술명과 별칭(alias) 사전 정의
2. **태그 추출** — 섹션(About the Role/Requirements류만 포함, Who we are/Salary류는 제외) 가중치를 적용해 description 열에서 실제 `skills` 태그 추출

부록 (사전/로직을 직접 손볼 때만 수동 실행):

- **A. 기술스택 후보 마이닝** — 사전에 없는 새 기술이 등장했는지 확인하고 싶을 때
- **B. 회귀 테스트** — 사전이나 매칭 로직을 고친 뒤 예전 버그가 재발하지 않았는지 확인하고 싶을 때


## 왜 사전(dictionary) 기반인가

NER 모델을 사용하는 것은 파이프라인에 들어가기에는 좀 무겁고 시간이 오래 걸려 병목현상이
발생할 수 있다고 판단했다. BERT와 같은 파이프라인에 들어갈만한 모델도 사용해보았으나 기술
스택과 연관이 없는 단어들을 많이 추출했다. 그래서 자체 제작한 기술스택 사전을 기반으로
description에서 정규식 매칭으로 기술 스택을 추출하는 방식을 택했다.

In [ ]:
import re
import difflib
from collections import Counter

import pandas as pd


## 0. 중복 공고 제거

앞으로 채용 데이터를 계속 새로 수집할 예정이므로, 두 단계로 중복을 제거해둔다.

1. **완전 중복**: description이 100% 동일한 공고는 하나만 남긴다.
2. **근접 중복**: `company`+`title`이 같은 그룹 안에서, 유사도(difflib) 0.95 이상인 공고도 하나만 남긴다.
   확인해보니 이런 경우 실제로는 같은 포지션을 US$/GBP처럼 통화만 바꿔 재게시한 것이었다
   (예: Anthropic Kubernetes Platform 공고가 `$40... USD` vs `£32... GBP`로만 다름, 유사도 0.999).
   차이 나는 부분도 대부분 Salary 섹션이라 skills 추출에는 어차피 영향이 없다.

단, `company`+`title`만 같고 실제 내용은 많이 다른 공고(유사도 0.95 미만)는 별개 포지션일 가능성이 높아
그대로 둔다 — 이전에 확인한 Airbnb "Senior Staff Software Engineer, Payments" 사례(유사도 0.147, 하나는
일반 Payments 엔지니어링, 하나는 Payments Compliance/KYC 전담)처럼 실제로 다른 포지션인 경우가 있었다.

원본은 `dev_role_jobs_raw_backup.csv`로 백업한 뒤 `dev_role_jobs.csv`를 덮어쓴다.

**재실행 가드**: 이 셀을 이미 처리된 `dev_role_jobs.csv`에 대해 다시 돌리면(수동 재실행, 자동화 재시도 등),
원래는 그 시점의 (이미 중복 제거된) 내용을 그대로 백업에 복사해버려서 "중복 제거 전 원본" 백업이
"중복 제거된 사본"으로 덮어써지는 문제가 있었다. 결과 파일(`dev_role_jobs.csv`) 자체는 dedup이
멱등이라 안 바뀌지만, 백업만 조용히 오염되고 티가 안 나서 나중에 진짜 원본이 필요할 때(예: dedup 로직
버그를 뒤늦게 발견해 처음부터 재처리해야 할 때) 복구할 수 없게 된다. 그래서 완전 중복이 하나도 없는
입력(=이미 처리된 파일)이 들어오면 백업 갱신과 dedup 실행을 모두 건너뛰도록 가드를 추가했다.


In [ ]:
raw_csv = "dev_role_jobs.csv"
raw_backup_csv = "dev_role_jobs_raw_backup.csv"
dedup_col = "description"
group_cols = ["company", "title"]
near_dup_threshold = 0.95

df_raw = pd.read_csv(raw_csv)
before = len(df_raw)

# 가드: dedup 결과물은 완전 중복이 항상 0건이 되도록 설계돼 있으므로(drop_duplicates),
# raw_csv에 완전 중복이 하나도 없으면 이미 이 셀로 처리된 파일이 재입력된 것으로 본다.
# 이 경우 백업을 덮어쓰면 "중복 제거 전 원본"이 "중복 제거된 사본"으로 영구히 대체되므로
# 백업 갱신과 dedup 실행을 모두 건너뛴다.
exact_dup_count = df_raw.duplicated(subset=[dedup_col]).sum()

if exact_dup_count == 0:
    print(f"{raw_csv}에 완전 중복이 없습니다 — 이미 중복 제거된 파일로 보여 이번 실행을 건너뜁니다.")
    print(f"({raw_backup_csv} 백업은 건드리지 않았습니다.) 새로 수집한 원본 데이터로 교체한 뒤 다시 실행하세요.")
else:
    df_raw.to_csv(raw_backup_csv, index=False)

    # 1) description이 완전히 동일한 공고 제거
    df_dedup = df_raw.drop_duplicates(subset=[dedup_col], keep="first")
    after_exact = len(df_dedup)

    # 2) company+title이 같은 그룹 안에서, 이미 남긴 것과 유사도 0.95 이상인 근접 중복 제거
    #    (예: 같은 포지션인데 통화만 다르게 올라온 US$/GBP 버전)
    keep_mask = pd.Series(True, index=df_dedup.index)
    for _, idx in df_dedup.groupby(group_cols).groups.items():
        idx = list(idx)
        if len(idx) < 2:
            continue
        kept_texts = []
        for i in idx:
            text = df_dedup.loc[i, dedup_col]
            is_near_dup = any(
                difflib.SequenceMatcher(None, text, kt).ratio() >= near_dup_threshold
                for kt in kept_texts
            )
            if is_near_dup:
                keep_mask[i] = False
            else:
                kept_texts.append(text)

    df_dedup = df_dedup[keep_mask]
    after = len(df_dedup)

    df_dedup.to_csv(raw_csv, index=False)

    print(f"완전 중복 {before - after_exact}건 제거 ({before:,} -> {after_exact:,}건)")
    print(f"근접 중복(유사도 {near_dup_threshold} 이상, 같은 company+title 안에서만) {after_exact - after}건 추가 제거 ({after_exact:,} -> {after:,}건)")
    print(f"원본은 {raw_backup_csv} 로 백업")


## 1. 기술 스택 사전 (`tech_dictionary.py`)

구조: `category -> { canonical_name: [alias1, alias2, ...] }`
- `canonical_name`은 매칭 후 tags 열에 실제로 들어갈 표준 이름.
- `aliases`에는 canonical_name 자체를 포함해서 적어야 함(자동으로 추가되지 않음).
- 대소문자/표기 변형(JS, Javascript 등)은 alias로 추가.

한 글자~두 글자짜리 애매한 언어명(Go, R, C, D)은 `AMBIGUOUS_EXACT`에 등록해서
다음 단계(`extract_tech_tags`)에서 대소문자 구분 + 엄격한 경계로만 매칭한다.
이 사전은 시작점일 뿐이며, 2단계에서 뽑은 후보를 검토해서 계속 늘려가는 것을 전제로 한다.


## 사전 구성 방식 (4단계 하이브리드)

1. **사전지식 기반 시드** — 일반적으로 알려진 기술(언어/프론트엔드/백엔드/DB/클라우드·DevOps/데이터·ML/모바일/툴 등)을 카테고리별로 수동 정리.
2. **애매한 짧은 토큰 분리** — `Go`/`R`/`C`/`D`처럼 흔한 영단어와 겹치는 1~2글자 언어명은 `AMBIGUOUS_EXACT`로 분리해서, 대소문자 구분 + 주변에 다른 확실한 기술어가 있어야만 인정.
3. **실제 데이터 기반 확장** — `mine_tech_candidates`로 description에서 사전에 없는 후보를 마이닝하고(회사명·일반 영단어 자동 필터링), 사람이 직접 골라 사전에 추가.
4. **실제 오탐 사례로 반복 보정** — 파이프라인을 돌려보며 발견한 진짜 오탐(`C++`/`C#` 경계 문제, `D.C.`/`WE'D`/`Series C`/`R&D`/`Go to Market` 등 ambiguous 토큰 오매칭, 회사가 자기 제품을 소개하며 생기는 자기언급 오탐)을 하나씩 규칙으로 보정.

## 매칭 로직 핵심 설계

- **경계 처리**: `C++`, `C#`, `.NET`처럼 특수문자가 낀 토큰은 기본 `\b`가 깨지므로 커스텀 경계 문자 클래스 사용. 아포스트로피·각종 대시·`&`/`/`/`.`/`+`/`#`까지 "단어의 일부"로 취급해서 `D.C.`, `WE'D`, `C‑suite`(특수 대시), `Series A–C` 같은 오매칭 방지. (`+`/`#`은 회귀 테스트로 뒤늦게 발견 — 없었을 땐 `"C++"`이라고만 쓰인 문장에서 `C++`(normal 패턴)과 별개로 `C`(ambiguous 패턴)까지 중복으로 잡혀서, 실제 데이터의 `C` 태그 238건 중 211건이 오탐이었음.)
- **긴 alias 우선 매칭**: `JavaScript`를 `Java`보다 먼저 시도해서 `JavaScript` 안의 `Java`를 잘못 떼어내지 않음.
- **ambiguous 토큰**(`Go`/`R`/`C`/`D`): 대소문자 구분 + 매칭 주변 50자 이내에 다른(비-ambiguous) 기술어가 있어야만 채택.
- **섹션 가중치**: `About the Role`/`Responsibilities`/`Requirements`/`Qualifications`류 섹션만 포함하고, `Who we are`/`Salary`/`Benefits`류 보일러플레이트는 제외. 헤더가 하나도 인식 안 되는 공고는 원문 전체로 폴백. (Tier 1·2 구분 없이 이진 포함/제외이며 가중치 차등은 없음 — 의도적 단순화.)
- **회사 자기언급 필터**: 태그가 회사명과 **정확히 같을 때만** 제거 후보로 보고, ambiguous 토큰과 같은 방식으로 매칭 주변 50자 이내에 다른 기술어가 있으면(=실제 스택 나열의 일부일 가능성이 높으면) 예외적으로 유지한다. `MongoDB Atlas`/`Cloudflare Workers`처럼 회사명을 포함하지만 canonical 이름 자체가 다른 태그는 애초에 자기언급 후보가 아니라 항상 유지된다.
  - 원래는 "태그가 회사명 전체/앞단어를 포함하거나 그 반대"면 부분일치로 무조건 제거했으나, 검증해보니 이 부분일치 규칙 때문에 Cloudflare 자사 공고에서 "Cloudflare" 태그 133건 중 131건이 삭제되고, 그 여파로 별개 제품명인 "Cloudflare Workers"까지 20건 중 19건이 같이 삭제되는 문제가 있었다. 131건을 직접 검토해보니 그중 실제로 주변에 다른 기술어와 함께 등장하는(=진짜 스택 요구일 가능성이 높은) 경우는 12건뿐이라, 부분일치를 정확히 일치로 좁히고 나머지는 ambiguous 토큰과 동일한 문맥 판단으로 정밀화했다.
- **스킬 별칭 병합**(`_SKILL_ALIAS_MERGE`): 사전에서는 "GPT"와 "OpenAI"를 별개 canonical로 뽑는다 — 회사 자기언급 필터가 회사별로 독립적으로 작동하려면 매칭 단계에서는 분리돼 있어야 하기 때문. 하지만 프론트가 참조하는 생태계 목록(27개)은 GPT/OpenAI를 하나로 묶어 두므로, 자기언급 필터까지 끝난 뒤 최종 태그 단계에서 "OpenAI" → "GPT"로 병합한다.
- **개념/실천법 용어 의도적 제외**: `LLM`, `RAG`, `DevOps`, `MLOps`, `NoSQL`, `Web3` 등은 "배우는 구체적 도구"가 아니라 범주/실천법이라 태깅하지 않기로 결정. (`NoSQL`의 하위 개념인 `MongoDB` 등은 정상적으로 태깅됨)

In [ ]:
TECH_DICTIONARY = {
    "language": {
        "Python": ["Python"],
        "Java": ["Java"],
        "JavaScript": ["JavaScript", "Javascript", "JS"],
        "TypeScript": ["TypeScript", "Typescript", "TS"],
        "Go": ["Golang", "Go"],
        "Rust": ["Rust"],
        "C++": ["C++"],
        "C#": ["C#"],
        "C": ["C"],
        "Ruby": ["Ruby"],
        "PHP": ["PHP"],
        "Swift": ["Swift"],
        "Kotlin": ["Kotlin"],
        "Scala": ["Scala"],
        "R": ["R"],
        "Objective-C": ["Objective-C", "Objective C", "ObjC"],
        "Dart": ["Dart"],
        "Elixir": ["Elixir"],
        "Erlang": ["Erlang"],
        "Haskell": ["Haskell"],
        "Perl": ["Perl"],
        "MATLAB": ["MATLAB", "Matlab"],
        "SQL": ["SQL"],
        "Bash": ["Bash", "Bash scripting", "Shell scripting", "Shell"],
        "Julia": ["Julia"],
        "Groovy": ["Groovy"],
        "Lua": ["Lua"],
        "Solidity": ["Solidity"],
        "D": ["D"],
        "Assembly": ["Assembly"],
        "SystemVerilog": ["SystemVerilog"],
        "VBA": ["VBA"],
        "SAS": ["SAS"],
        "Zig": ["Zig"],
        "GLSL": ["GLSL"],
        "Cypher": ["Cypher"],
    },
    "frontend": {
        "React": ["React", "React.js", "ReactJS"],
        "Angular": ["Angular", "AngularJS", "Angular.js"],
        "Vue.js": ["Vue.js", "Vue", "VueJS"],
        "Svelte": ["Svelte", "SvelteKit"],
        "Next.js": ["Next.js", "NextJS"],
        "Nuxt.js": ["Nuxt.js", "NuxtJS"],
        "jQuery": ["jQuery"],
        "Redux": ["Redux"],
        "HTML": ["HTML", "HTML5"],
        "CSS": ["CSS", "CSS3"],
        "Sass": ["Sass", "SCSS"],
        "Tailwind CSS": ["Tailwind CSS", "TailwindCSS", "Tailwind"],
        "Webpack": ["Webpack"],
        "Vite": ["Vite"],
        "ESLint": ["ESLint"],
        "Babel": ["Babel"],
        "Ember.js": ["Ember.js", "EmberJS"],
        "Backbone.js": ["Backbone.js", "BackboneJS"],
        "D3.js": ["D3.js", "D3JS"],
        "Storybook": ["Storybook"],
        "MobX": ["MobX"],
        "SolidJS": ["SolidJS", "Solid.js"],
        "styled-components": ["styled-components"],
        "Gatsby": ["Gatsby"],
        "TanStack": ["TanStack"],
        "Material UI": ["Material UI", "MUI"],
    },
    "backend": {
        "Node.js": ["Node.js", "NodeJS", "Node"],
        "Express": ["Express.js", "ExpressJS", "Express"],
        "Django": ["Django"],
        "Flask": ["Flask"],
        "FastAPI": ["FastAPI"],
        "Spring": ["Spring Boot", "Spring Framework", "SpringBoot"],
        "Ruby on Rails": ["Ruby on Rails", "Rails"],
        "Laravel": ["Laravel"],
        ".NET": [".NET", "ASP.NET", "dotnet"],
        "NestJS": ["NestJS", "Nest.js"],
        "Deno": ["Deno"],
        "gRPC": ["gRPC"],
        "GraphQL": ["GraphQL"],
        "REST": ["REST API", "RESTful"],
    },
    "database": {
        "PostgreSQL": ["PostgreSQL", "Postgres"],
        "MySQL": ["MySQL"],
        "MongoDB": ["MongoDB", "Mongo"],
        "Redis": ["Redis"],
        "Cassandra": ["Cassandra"],
        "DynamoDB": ["DynamoDB"],
        "Elasticsearch": ["Elasticsearch", "ElasticSearch"],
        "SQLite": ["SQLite"],
        "Oracle DB": ["Oracle Database", "Oracle DB"],
        "MariaDB": ["MariaDB"],
        "Snowflake": ["Snowflake"],
        "BigQuery": ["BigQuery"],
        "Redshift": ["Redshift"],
        "ClickHouse": ["ClickHouse"],
        "Neo4j": ["Neo4j"],
        "CockroachDB": ["CockroachDB"],
        "Firestore": ["Firestore"],
        "MongoDB Atlas": ["MongoDB Atlas"],
    },
    "cloud_devops": {
        "AWS": ["AWS", "Amazon Web Services"],
        "GCP": ["GCP", "Google Cloud Platform", "Google Cloud"],
        "Azure": ["Azure", "Microsoft Azure"],
        "Docker": ["Docker"],
        "Kubernetes": ["Kubernetes", "K8s"],
        "Terraform": ["Terraform"],
        "Ansible": ["Ansible"],
        "Jenkins": ["Jenkins"],
        "GitHub Actions": ["GitHub Actions"],
        "GitLab CI": ["GitLab CI", "GitLab CI/CD"],
        "CircleCI": ["CircleCI"],
        "Helm": ["Helm"],
        "Istio": ["Istio"],
        "Prometheus": ["Prometheus"],
        "Grafana": ["Grafana"],
        "Datadog": ["Datadog"],
        "Nginx": ["Nginx"],
        "Linux": ["Linux"],
        "Cloudflare": ["Cloudflare"],
        "Vault": ["HashiCorp Vault", "Vault"],
        "Cloudflare Workers": ["Cloudflare Workers"],
        "Envoy": ["Envoy"],
        "Heroku": ["Heroku"],
        "Fly.io": ["Fly.io"],
        "Vercel": ["Vercel"],
        "Kibana": ["Kibana"],
        "TeamCity": ["TeamCity"],
        "IIS": ["IIS"],
        "OpenTelemetry": ["OpenTelemetry"],
        "AWS CloudFormation": ["CloudFormation"],
        "Amazon EKS": ["EKS", "Amazon EKS"],
        "Google GKE": ["GKE", "Google Kubernetes Engine"],
        "Amazon EC2": ["EC2", "Amazon EC2"],
        "Amazon S3": ["S3", "Amazon S3"],
        "ArgoCD": ["ArgoCD", "Argo CD"],
        "Buildkite": ["Buildkite"],
        "OpenShift": ["OpenShift"],
        "Pulumi": ["Pulumi"],
        "Dependabot": ["Dependabot"],
        "Puppet": ["Puppet"],
        "Snyk": ["Snyk"],
        "Okta": ["Okta"],
        "Splunk": ["Splunk"],
        "WebAssembly": ["WebAssembly", "Wasm"],
        "Bicep": ["Bicep"],
        "Starlark": ["Starlark"],
        "Nix": ["Nix"],
    },
    "data_ml": {
        "TensorFlow": ["TensorFlow"],
        "PyTorch": ["PyTorch"],
        "Keras": ["Keras"],
        "scikit-learn": ["scikit-learn", "sklearn"],
        "Pandas": ["Pandas"],
        "NumPy": ["NumPy"],
        "Apache Spark": ["Apache Spark", "Spark", "PySpark"],
        "Hadoop": ["Hadoop"],
        "Apache Kafka": ["Apache Kafka", "Kafka"],
        "Airflow": ["Airflow", "Apache Airflow"],
        "dbt": ["dbt"],
        "Databricks": ["Databricks"],
        "Jupyter": ["Jupyter", "Jupyter Notebook"],
        "Hugging Face": ["Hugging Face", "HuggingFace"],
        "Transformers": ["Transformers"],
        "LangChain": ["LangChain"],
        "OpenCV": ["OpenCV"],
        "MLflow": ["MLflow"],
        "XGBoost": ["XGBoost"],
        "LightGBM": ["LightGBM"],
        "Tableau": ["Tableau"],
        "Power BI": ["Power BI", "PowerBI"],
        "Looker": ["Looker"],
        "Streamlit": ["Streamlit"],
        "Plotly": ["Plotly"],
        "Lucene": ["Lucene", "Apache Lucene"],
        "Apache Flink": ["Apache Flink", "Flink"],
        "Apache Iceberg": ["Apache Iceberg", "Iceberg"],
        "Trino": ["Trino"],
        "Dagster": ["Dagster"],
        "Vertex AI": ["Vertex AI", "Vertex"],
        "Triton Inference Server": ["Triton"],
        "JAX": ["JAX"],
        "CUDA": ["CUDA"],
        "NCCL": ["NCCL"],
        "GitHub Copilot": ["Copilot", "GitHub Copilot"],
        "OpenAI Codex": ["Codex"],
        "MCP": ["MCP", "Model Context Protocol"],
        "Slurm": ["Slurm"],
        "Bazel": ["Bazel"],
        "Gradle": ["Gradle"],
        "Mojo": ["Mojo"],
    },
    "mobile": {
        "iOS": ["iOS"],
        "Android": ["Android"],
        "React Native": ["React Native"],
        "Flutter": ["Flutter"],
        "SwiftUI": ["SwiftUI"],
        "Xamarin": ["Xamarin"],
        "Jetpack Compose": ["Jetpack Compose"],
        "UIKit": ["UIKit"],
    },
    "messaging_infra": {
        "RabbitMQ": ["RabbitMQ"],
        "Apache Pulsar": ["Apache Pulsar", "Pulsar"],
        "gRPC": ["gRPC"],
        "Zookeeper": ["Zookeeper"],
    },
    "tools": {
        "Git": ["Git"],
        "GitHub": ["GitHub"],
        "GitLab": ["GitLab"],
        "Jira": ["Jira"],
        "Figma": ["Figma"],
        "Postman": ["Postman"],
        "Confluence": ["Confluence"],
        "GDB": ["GDB"],
        "Salesforce": ["Salesforce", "Salesforce.com", "SFDC"],
        "HubSpot": ["HubSpot", "Hubspot"],
        "Zendesk": ["Zendesk"],
        "NetSuite": ["NetSuite", "Netsuite"],
        "Vanta": ["Vanta"],
    },
    "ai_platform": {
        "OpenAI": ["OpenAI"],
        "Anthropic": ["Anthropic"],
        "Claude": ["Claude"],
        "GPT": ["GPT", "GPT-4", "ChatGPT"],
        "Gemini": ["Gemini"],
        "Llama": ["Llama"],
    },
    "testing": {
        "Jest": ["Jest"],
        "PyTest": ["PyTest", "pytest"],
        "Selenium": ["Selenium"],
        "Cypress": ["Cypress"],
        "JUnit": ["JUnit"],
    },
}

# 대소문자를 반드시 구분해서, "엄격한" 경계로만 매칭할 애매한 짧은 토큰.
# 예: "Go" vs 일반 동사 "go", "R" vs "R&D"의 R, "C" vs 성적 등급 C.
# 완벽하지 않으니 추출 후 결과를 한 번 훑어보고 오탐이 많으면 여기서 빼거나
# extract_tech_tags 의 매칭 규칙을 더 강화할 것.
AMBIGUOUS_EXACT = {"Go", "R", "C", "D"}


## 외부 자료로 사전 검증

정답 라벨이 없어서 정밀도/재현율을 직접 계산할 순 없지만, 공신력 있는 외부 자료와 대조해서 사전의
타당성을 검증했다. 세 자료 모두 "빈도만 보고 바로 추가"하지 않고 실제 corpus 문맥을 확인한 뒤
채택 여부를 결정했다 — 공식 자료라도 흔한 영단어와 겹치는 이름(예: `Render`, `Play`, `Kong`)은
오탐일 수 있기 때문.

- **[GitHub Linguist](https://github.com/github-linguist/linguist)** (`languages.yml`): 프로그래밍 언어 557개 중 사전에 없던 것을 대조 → 실제 데이터에 등장하는 67개 확인 → 문맥 검증 후 13개 채택 (`Assembly`, `SystemVerilog`, `VBA`, `SAS`, `Zig`, `GLSL`, `Cypher`, `Mojo`, `Bicep`, `Starlark`, `Nix`, `GDB`, `Shell`). `B`, `Self`, `Click`, `Processing`, `Fluent` 등은 일반 영단어 오탐으로 확인되어 제외.
- **[Wappalyzer](https://github.com/enthec/webappanalyzer)** (오픈소스 기술 감지 DB): 개발자 핵심 카테고리(언어/프레임워크/DB/CI/컨테이너/PaaS/IaaS 등)로 좁혀 878개 대조 → 실제 등장 41개 확인 → 문맥 검증 후 19개 채택 (`Vercel`, `Envoy`, `Heroku`, `Kibana`, `Streamlit`, `UIKit`, `Deno` 등). 애매했던 9개(`Render`, `Railway`, `Apollo`, `Prototype`, `AMP`, `Play`, `Warp`, `Kong`, `Bootstrap`)는 전부 오탐으로 확인되어 제외 — 대부분 회사 자기소개(Render/Railway 자체가 이 데이터셋의 채용 회사)이거나 일반 영단어 용법이었음.
- **[Lightcast Skills Extractor](https://lightcast.io/open-skills/extraction)**: 실제 공고 2건을 붙여넣어 결과 비교 → 우리가 뽑은 태그 전부 대응 확인(정밀도 검증 통과), `ESLint`/`Babel` 누락을 발견해서 추가. Lightcast는 `Blockchain`, `Web 3.0`, `UX/UI`, `Innovation` 같은 개념어·소프트스킬도 태깅하는데, 이는 우리가 의도적으로 뺀 범주와 정확히 겹쳐서 스코프 결정이 합리적이었음을 재확인. Lightcast 쪽에도 원문에 없는 `Cross-Site Request Forgery`, `Microsoft Teams` 같은 오탐이 있어 이 문제가 우리만의 한계가 아님을 확인.
- **Lightcast Open Skills Taxonomy 자체**는 대량 대조에는 못 씀 — 페이지가 JS로 동적 로딩되는 구조라 정적 fetch로 못 읽고, 대량 API는 계약 기반 액세스 요청이 필요함.

## 2. 기술 태그 추출

description 열에서 1단계 사전을 기반으로 기술 스택을 뽑아
`skills` 열(콤마로 구분된 canonical name 문자열)로 저장한다.

전체 텍스트를 다 보지 않고, `About the Role`/`Responsibilities`/`Requirements`/`Qualifications`류
섹션만 포함하고 `Who we are`/`Benefits`/`Salary`류 보일러플레이트 섹션은 제외한다
(섹션 헤더가 하나도 인식 안 되는 공고는 원문 전체로 폴백). 회사가 자기 이름을 소개 문단에서
언급하며 생기는 자기언급 오탐(예: Cloudflare 자사 소개에 "Cloudflare"가 반복 등장)도,
주변에 다른 기술어가 없을 때만 걸러낸다.


In [ ]:
# 일반 토큰의 경계: 문자/숫자/밑줄이 아니면 경계로 취급 (대소문자 무시 매칭)
_WORD = r"A-Za-z0-9_"
# ambiguous 토큰(Go, R, C, D)의 경계는 더 엄격하게 잡는다.
# "R&D"의 R, "A/B"의 B, "C-suite"/"C‑suite"(특수 대시)의 C, "Series A–C"의 C(en dash),
# "D.C."의 D, "WE'D"/"WE'D"(아포스트로피 축약형)의 D, "C++"/"C#"의 C(회귀 테스트로 발견:
# ambiguous 패턴과 normal 패턴은 서로 독립적인 별개 스캔이라, +/#이 경계 문자에 없으면
# "C++"에서 "C++"(normal)과 별개로 "C"(ambiguous)까지 중복으로 잡힌다) 처럼
# 약어·합성어·축약형 안에 낀 한 글자를 오매칭하지 않도록 &, /, ., 아포스트로피, 각종 대시,
# +, # 도 "단어의 일부"로 취급해서 제외한다.
_WORD_EXT = r"A-Za-z0-9_&/.'’‐‑‒–—―+#\-"

# 그래도 남는 "Go to Market", "Series C" 같은 오탐은 문자 경계만으로 못 거른다.
# 실제 기술스택 나열은 거의 항상 다른(비-ambiguous) 기술명과 가까이 등장하므로,
# ambiguous 매칭 주변 이 범위 안에 정상 매칭이 하나도 없으면 버린다.
# 회사 자기언급 필터(아래 extract_tags)도 같은 창을 재사용한다.
_AMBIGUOUS_CONTEXT_WINDOW = 50


def _flatten_dictionary():
    normal_lookup = {}  # alias.lower() -> canonical
    ambiguous_lookup = {}  # alias(대소문자 그대로) -> canonical
    normal_aliases = []
    ambiguous_aliases = []
    for canonical_map in TECH_DICTIONARY.values():
        for canonical, aliases in canonical_map.items():
            for alias in aliases:
                if alias in AMBIGUOUS_EXACT:
                    ambiguous_lookup[alias] = canonical
                    ambiguous_aliases.append(alias)
                else:
                    normal_lookup[alias.lower()] = canonical
                    normal_aliases.append(alias)
    return normal_lookup, ambiguous_lookup, normal_aliases, ambiguous_aliases


def _build_pattern(aliases, boundary_class, flags=0):
    # 같은 위치에서 더 구체적인(긴) 표현을 먼저 시도하도록 길이 내림차순 정렬
    # 예: "JavaScript"를 "Java"보다 먼저 시도해야 "JavaScript" 안의 "Java"를 잘못 떼어내지 않음
    aliases_sorted = sorted(set(aliases), key=len, reverse=True)
    escaped = [re.escape(a) for a in aliases_sorted]
    left = rf"(?<![{boundary_class}])"
    right = rf"(?![{boundary_class}])"
    return re.compile(left + "(?:" + "|".join(escaped) + ")" + right, flags)


_NORMAL_LOOKUP, _AMBIGUOUS_LOOKUP, _NORMAL_ALIASES, _AMBIGUOUS_ALIASES = _flatten_dictionary()
_NORMAL_PATTERN = _build_pattern(_NORMAL_ALIASES, _WORD, flags=re.IGNORECASE)
_AMBIGUOUS_PATTERN = _build_pattern(_AMBIGUOUS_ALIASES, _WORD_EXT)

# 업무 서술(About the Role/Responsibilities류)과 명시적 요건(Requirements/Qualifications류)
# 섹션 헤더. 구분 없이 하나의 집합으로 취급하며, 둘 다 "실제로 쓰는 기술"이 문장 속에
# 자연스럽게 등장할 확률이 높아서 추출 대상에 포함한다(이진 포함/제외이며 가중치 차등은 없음).
_INCLUDE_HEADERS = {
    "about the role", "about this role", "the role", "about the team",
    "responsibilities", "key responsibilities", "core responsibilities",
    "what you'll do", "what you will do", "in this role, you will", "you will",
    "a typical day", "the difference you will make",
    "requirements", "minimum requirements", "minimum qualifications", "preferred qualifications",
    "qualifications", "your expertise", "experience", "bonus points", "nice to have", "nice to haves",
    "who you are", "you have", "you are", "skills",
    "you might thrive in this role if you", "you may be a good fit if you",
    "you may be a good fit if you have", "what we're looking for",
    "what you'll bring", "what you will bring",
}

# 회사 소개/복지/급여/정책류 보일러플레이트. 기술 언급이 거의 없고
# 회사 자기소개 문단(자기언급 오탐의 원인) 대부분이 여기 들어간다.
_EXCLUDE_HEADERS = {
    "a world-changing company", "about us", "benefits", "who we are", "how we're different",
    "come work with us", "life at palantir", "what makes cloudflare special", "why binance",
    "equal opportunity at datadog", "privacy and ai guidelines", "compensation", "salary",
    "salary range", "pay range", "base salary", "what we offer", "benefits and growth",
    "who are we", "how and where we work", "mental health benefits", "family building benefits",
    "child care and pet benefits", "equity", "dental insurance", "who may apply",
    "equal employment opportunity", "diversity", "perks", "visa sponsorship",
    "work authorization", "how to apply", "application process",
}

_HEADER_STRIP_CHARS = ":?!. "


def _normalize_header(line):
    return line.strip().strip(_HEADER_STRIP_CHARS).lower()


def section_filtered_text(text):
    """
    'Who we are/Benefits/Salary'류 보일러플레이트 섹션은 버리고,
    'About the Role/Responsibilities/Requirements'류 섹션 내용만 이어붙여 반환한다.
    인식되는 헤더가 하나도 없는 공고(포맷이 다른 경우)는 원문 전체로 폴백한다.
    """
    if not isinstance(text, str) or not text:
        return text

    state = "exclude"  # 첫 헤더가 나오기 전 도입부는 보통 회사 소개라 기본적으로 제외
    seen_header = False
    kept = []
    for line in text.split("\n"):
        key = _normalize_header(line)
        if key in _INCLUDE_HEADERS:
            state, seen_header = "include", True
            continue
        if key in _EXCLUDE_HEADERS:
            state, seen_header = "exclude", True
            continue
        if key.startswith("about ") and key not in ("about the role", "about this role", "about the team"):
            # "About OpenAI" / "About MongoDB" 처럼 회사명이 들어간 소개 섹션
            state, seen_header = "exclude", True
            continue
        if state == "include":
            kept.append(line)

    filtered = "\n".join(kept).strip()
    if not seen_header or not filtered:
        return text
    return filtered


def extract_tags(text, company=None):
    """
    텍스트 하나에서 canonical 기술명 리스트(정렬, 중복 제거)를 반환.

    company가 주어지면, 채용 회사 이름과 정확히 같은 canonical 기술명(예: Cloudflare
    채용공고에서의 "Cloudflare")은 매칭 주변 50자 안에 다른 기술 태그가 없을 때만
    회사 자기소개성 언급으로 보고 제외한다. "Cloudflare Workers"/"MongoDB Atlas"처럼
    회사명을 포함하지만 canonical 이름 자체가 다른 태그는 애초에 자기언급 후보가
    아니므로 항상 유지된다.
    """
    if not isinstance(text, str) or not text:
        return []
    found = set()
    company_lower = company.strip().lower() if isinstance(company, str) and company.strip() else None

    normal_spans = list(_NORMAL_PATTERN.finditer(text))
    self_mention_spans = []
    other_spans = []
    for m in normal_spans:
        canonical = _NORMAL_LOOKUP[m.group().lower()]
        if company_lower and canonical.lower() == company_lower:
            self_mention_spans.append((m, canonical))
        else:
            found.add(canonical)
            other_spans.append(m)

    for m, canonical in self_mention_spans:
        s, e = m.span()
        window_start, window_end = s - _AMBIGUOUS_CONTEXT_WINDOW, e + _AMBIGUOUS_CONTEXT_WINDOW
        has_nearby_tech = any(
            n.start() < window_end and n.end() > window_start for n in other_spans
        )
        if has_nearby_tech:
            found.add(canonical)

    for m in _AMBIGUOUS_PATTERN.finditer(text):
        s, e = m.span()
        window_start, window_end = s - _AMBIGUOUS_CONTEXT_WINDOW, e + _AMBIGUOUS_CONTEXT_WINDOW
        has_nearby_tech = any(
            n.start() < window_end and n.end() > window_start for n in normal_spans
        )
        if has_nearby_tech:
            found.add(_AMBIGUOUS_LOOKUP[m.group()])

    return sorted(found)


# 사전에서는 별개 canonical(자기언급 필터가 회사별로 독립적으로 작동해야 하므로)이지만,
# 최종 집계에서는 같은 스킬로 합쳐야 하는 쌍. 원래는 ERD의 SKILL_ALIAS 테이블(DB 레이어)로
# 옮기는 게 맞는 문제이나, DB 반영 전까지는 태그 추출 마지막 단계에서 우선 병합해둔다.
# 병합은 extract_tags(자기언급 필터 포함) 이후에 적용해야, OpenAI 자사 공고의 일반 언급이
# "회사명과 canonical 불일치"로 필터를 우회해 GPT로 새어 들어오는 걸 막을 수 있다.
_SKILL_ALIAS_MERGE = {"OpenAI": "GPT"}


def tag_dataframe(df, text_col="description", tag_col="skills", company_col="company"):
    """
    company_col이 주어지면 각 행의 company를 extract_tags에 넘겨서, 회사명과 정확히
    같은 canonical 기술명만 자기언급 후보로 보고 주변 문맥에 따라 제거한다.
    상세 규칙은 extract_tags 참고. 그 뒤 _SKILL_ALIAS_MERGE에 등록된 쌍을 하나로 합친다.
    """
    df = df.copy()
    has_company = bool(company_col) and company_col in df.columns

    def _row_tags(text, company):
        tags = extract_tags(section_filtered_text(text), company=company if has_company else None)
        tags = sorted({_SKILL_ALIAS_MERGE.get(t, t) for t in tags})
        return ", ".join(tags)

    if has_company:
        df[tag_col] = [
            _row_tags(t, c) for t, c in zip(df[text_col], df[company_col])
        ]
    else:
        df[tag_col] = df[text_col].apply(lambda t: _row_tags(t, None))
    return df


### 태그 추출 실행

1단계 사전을 기반으로 `skills` 열을 추가해 저장합니다.
(중복 제거는 0단계에서 이미 끝났으므로 여기서는 다시 하지 않습니다.)


In [ ]:
tag_input_csv = "dev_role_jobs.csv"
tag_output_csv = "dev_role_jobs_tagged.csv"
tag_text_col = "description"

df = pd.read_csv(tag_input_csv)

tagged = tag_dataframe(df, text_col=tag_text_col)
tagged.to_csv(tag_output_csv, index=False)

non_empty = (tagged["skills"] != "").sum()
print(f"{len(tagged):,}건 중 {non_empty:,}건({non_empty / len(tagged):.1%})에서 태그 추출됨")
print()
print("태그 빈도 Top 30")
print(tagged["skills"].str.split(", ").explode().value_counts().head(30))


## 부록 A. 기술스택 후보 마이닝 (선택 — 필요할 때만 수동 실행)

`TECH_DICTIONARY`에 아직 없는 "기술스택처럼 생긴" 토큰의 빈도를 뽑아서
사전을 늘려나갈 때 검토용 후보 리스트를 만든다.

사전이 이미 충분히 채워진 상태라 매번 자동으로 돌 필요는 없고,
자동 수집이 계속되면서 시간이 지나 사전에 없는 새 기술이 등장했는지 궁금할 때만 수동으로 돌리면 된다.
이 단계는 자동으로 사전에 추가하지 않는다. 출력된 후보를 사람이 훑어보고
진짜 기술스택만 골라 위 1단계 `TECH_DICTIONARY`에 직접 추가하는 것을 전제로 한다.


In [ ]:
# PostgreSQL, GraphQL, TensorFlow 같은 CamelCase / Node.js 같은 dotted 이름을 후보로 포착.
# 대문자로 시작 + 영숫자, 뒤에 ".js" 류 확장이나 "++","#" 접미사가 붙는 경우까지 허용.
CANDIDATE_PATTERN = re.compile(
    r"\b[A-Z][A-Za-z0-9]{1,}(?:\.[A-Za-z]{1,5})?(?:\+\+|#)?\b"
)
# candidate와 동일한 단어가 완전 소문자로도 등장하는 빈도를 재는 용도.
# "Build"/"Design"/"Learn" 처럼 불릿·문장 맨 앞이라 우연히 대문자가 된 일반 단어는
# 본문 다른 곳에서 소문자로도 흔히 쓰이지만, React/Kubernetes 같은 진짜 고유명사는 거의 항상 대문자로만 등장한다.
LOWERCASE_WORD_PATTERN = re.compile(r"\b[a-z]+\b")

# 채용공고에 아주 흔히 등장해서 후보로 나와봤자 노이즈인 단어들.
# 완전한 목록이 될 수 없으니 실제 출력을 보고 계속 추가해서 쓸 것.
STOPWORDS = {
    "The", "We", "Our", "You", "Your", "This", "That", "These", "Those", "As", "In", "At",
    "With", "For", "And", "Or", "But", "Is", "Are", "Was", "Were", "Be", "Been", "Being",
    "It", "Its", "If", "Then", "So", "Team", "Role", "About", "Who", "What", "When", "Where",
    "Why", "How", "Job", "Company", "Work", "Join", "Us", "Will", "Can", "May", "Must",
    "Should", "Would", "Could", "Please", "Note", "New", "All", "Most", "Some", "Any", "Each",
    "Every", "Other", "Also", "More", "Than", "Such", "Not", "No", "Yes", "Ok", "Well", "Good",
    "Great", "Strong", "Excellent", "Location", "Salary", "Benefits", "Requirements",
    "Qualifications", "Responsibilities", "Experience", "Skills", "Knowledge", "Ability",
    "Years", "Degree", "Bachelor", "Master", "Phd", "Apply", "Application", "Candidate",
    "Candidates", "Employer", "Employment", "Equal", "Opportunity", "Diversity", "Inclusion",
    "Benefit", "Health", "Insurance", "Remote", "Hybrid", "Onsite", "Office", "City", "State",
    "United", "States", "Global", "Global", "Department", "Manager", "Director", "Senior",
    "Junior", "Lead", "Head", "Level", "Full", "Part", "Time",
}


def known_aliases_lower():
    known = set()
    for canonical_map in TECH_DICTIONARY.values():
        for canonical, aliases in canonical_map.items():
            known.add(canonical.lower())
            for alias in aliases:
                known.add(alias.lower())
    return known


def mine(df, text_col="description", company_col="company", min_count=5, max_lowercase_ratio=0.3):
    known = known_aliases_lower()
    # 회사 소개/자사 언급 노이즈 제거: 이 데이터셋에 등장하는 채용 회사명 자체는 기술스택이 아니다.
    known_companies = set()
    if company_col and company_col in df.columns:
        known_companies = {c.lower() for c in df[company_col].dropna().unique()}

    counter = Counter()
    lowercase_counter = Counter()
    for text in df[text_col].dropna():
        for m in CANDIDATE_PATTERN.finditer(text):
            token = m.group()
            token_lower = token.lower()
            if token in STOPWORDS or token_lower in known or token_lower in known_companies:
                continue
            if len(token) < 2:
                continue
            counter[token] += 1
        for m in LOWERCASE_WORD_PATTERN.finditer(text):
            lowercase_counter[m.group()] += 1

    rows = []
    for tok, cnt in counter.items():
        if cnt < min_count:
            continue
        lower_cnt = lowercase_counter.get(tok.lower(), 0)
        if lower_cnt > max_lowercase_ratio * cnt:
            continue  # 소문자로도 흔히 쓰이는 일반 단어로 판단, 제외
        rows.append((tok, cnt, lower_cnt))

    rows.sort(key=lambda x: -x[1])
    return pd.DataFrame(rows, columns=["candidate", "count", "lowercase_count"])


### 후보 마이닝 실행

`input_csv`를 채용공고 데이터 CSV 경로로 바꿔서 실행하세요.


In [ ]:
input_csv = "dev_role_jobs.csv"
candidates_output_csv = "tech_candidates.csv"
text_col = "description"
min_count = 5

df = pd.read_csv(input_csv)
candidates = mine(df, text_col=text_col, min_count=min_count)
candidates.to_csv(candidates_output_csv, index=False)

print(f"후보 {len(candidates)}개를 {candidates_output_csv} 에 저장했습니다.")
print("csv파일 열어서 진짜 기술스택만 골라 1단계 TECH_DICTIONARY 에 추가하세요.")
print()
print(candidates.head(50).to_string(index=False))


## 부록 B. 회귀 테스트 (선택 — 사전/로직을 고쳤을 때만 수동 실행)

파이프라인을 만들며 실제로 겪었던 오탐/버그 사례들을 고정 케이스로 박아둔다.
사전이나 매칭 로직을 고칠 때마다 이 셀을 실행해서, 예전에 고친 문제가 조용히 재발하지 않았는지 확인한다.
메인 파이프라인(0~2단계)과는 독립적이라 로직을 안 건드리면 평소엔 안 돌려도 된다.

- **레이어 1**: `extract_tags` 정규식 매칭 자체 (경계 처리, ambiguous 토큰)
- **레이어 2**: `section_filtered_text` 섹션 필터링 (Who we are/Salary 제외, Requirements 포함)
- **레이어 3**: `tag_dataframe`의 회사 자기언급 필터
- **레이어 4**: `tag_dataframe`의 스킬 별칭 병합 (`_SKILL_ALIAS_MERGE`)

만들자마자 실제로 프로덕션 버그를 하나 잡아냈다: `"Proficient in C++, C#, and Python"` 케이스를 넣었더니
기대값(`C++`, `C#`, `Python`)과 다르게 `C`까지 같이 잡혔다. 위 "경계 처리" 항목에 적은 `+`/`#` 누락
버그였고, 고친 뒤 실제 데이터의 `C` 태그가 238건 → 27건으로 정정됐다(211건이 `C++`/`C#`에서 새어나온
오탐이었음). 손으로 스팟체크만 하던 기존 방식으로는 못 잡았을 버그라, 회귀 테스트가 "이미 아는 문제
재발 방지"뿐 아니라 "아직 몰랐던 문제 발견"에도 쓸모 있다는 걸 확인했다.


In [ ]:
# ============================================================
# 레이어 1: extract_tags 정규식 매칭
# ============================================================
layer1_cases = [
    ("Experience with JavaScript and Java", {"JavaScript", "Java"}),
    ("We use Golang and Go for backend, plus Rust", {"Go", "Rust"}),
    ("Go to Market strategy and business development", set()),
    ("Our last funding was a Series C round", set()),
    ("Washington, D.C. metro area required", set()),
    ("We'd love for you to join. WE'D really appreciate it", set()),
    ("R&D team focused on machine learning with Python", {"Python"}),
    ("Proficient in C++, C#, and Python", {"C++", "C#", "Python"}),
    ("Familiar with Node.js and .NET", {"Node.js", ".NET"}),
    ("Skilled in JavaScript, TypeScript, and modern JS frameworks like React", {"JavaScript", "TypeScript", "React"}),
    ("Strong C, C++, or Rust background required", {"C", "C++", "Rust"}),
    ("We use Kubernetes, AWS, and GCP for our infrastructure", {"Kubernetes", "AWS", "GCP"}),
]

# ============================================================
# 레이어 2: section_filtered_text (Who we are/Salary 제외, Requirements 포함)
# ============================================================
section_test_desc = """
Who we are

We are ExampleCorp, and our entire platform runs on MongoDB and Cloudflare.

Salary

The salary range for this role is listed in USD.

Requirements

Experience with Python and Kubernetes required.
"""

# ============================================================
# 레이어 3: 회사 자기언급 필터
# 회사명과 정확히 같은 태그만 후보로 보고, 주변 50자 안에 다른 기술어가 있으면 유지한다.
# ============================================================
mongodb_atlas_desc = "Requirements\n\nExperience with MongoDB Atlas required."
anthropic_short_tag_desc = "Requirements\n\nProficiency in C++, C required."
cloudflare_self_desc = "Requirements\n\nExperience with Cloudflare required."
cloudflare_context_desc = "Requirements\n\nExperience with Cloudflare, Kubernetes, and Python required."

layer3_cases = [
    ("MongoDB Atlas, company=MongoDB (canonical이 달라 자기언급 아님, 유지돼야 함)", mongodb_atlas_desc, "MongoDB", {"MongoDB Atlas"}),
    ("MongoDB Atlas, company=Stripe (자기언급 아님, 유지돼야 함)", mongodb_atlas_desc, "Stripe", {"MongoDB Atlas"}),
    ("C/C++ company=Anthropic (짧은 태그가 회사명 철자 때문에 사라지면 안 됨)", anthropic_short_tag_desc, "Anthropic", {"C", "C++"}),
    ("Cloudflare 고립된 언급, company=Cloudflare (주변에 다른 기술 없어 제거돼야 함)", cloudflare_self_desc, "Cloudflare", set()),
    ("Cloudflare, company=Cloudflare, 주변에 다른 기술 있음 (유지돼야 함)", cloudflare_context_desc, "Cloudflare", {"Cloudflare", "Kubernetes", "Python"}),
]

# ============================================================
# 레이어 4: 스킬 별칭 병합 (GPT/OpenAI)
# 매칭 단계에서는 별개 canonical이지만, 최종 출력에서는 "OpenAI"가 "GPT"로 합쳐져야 한다.
# 단, OpenAI 자사 공고의 자기언급 필터는 병합 전 canonical("OpenAI")로 정상 작동해야 한다.
# ============================================================
gpt_openai_desc = "Requirements\n\nExperience with GPT-4 and OpenAI required."
openai_self_isolated_desc = "Requirements\n\nExperience with OpenAI required."
openai_self_context_desc = "Requirements\n\nExperience with OpenAI, Kubernetes, and Python required."

layer4_cases = [
    ("GPT-4 + OpenAI, company 없음 (하나로 병합돼야 함)", gpt_openai_desc, None, {"GPT"}),
    ("OpenAI 고립 언급, company=OpenAI (자기언급 필터가 먼저 제거해야 함)", openai_self_isolated_desc, "OpenAI", set()),
    ("OpenAI+문맥, company=OpenAI (자기언급 필터 통과 후 GPT로 병합돼야 함)", openai_self_context_desc, "OpenAI", {"GPT", "Kubernetes", "Python"}),
]

# ============================================================
# 실행
# ============================================================
failed = 0

print("레이어 1: extract_tags 정규식 매칭")
for text, expected in layer1_cases:
    result = set(extract_tags(text))
    ok = result == expected
    failed += not ok
    print(f"  {'PASS' if ok else 'FAIL'}  {text[:55]!r}")
    if not ok:
        print(f"        기대: {expected} / 실제: {result}")

print()
print("레이어 2: section_filtered_text 섹션 필터링")
test_df = pd.DataFrame({"description": [section_test_desc], "company": ["ExampleCorp"]})
result = set(tag_dataframe(test_df).loc[0, "skills"].split(", ")) - {""}
expected = {"Python", "Kubernetes"}
ok = result == expected
failed += not ok
print(f"  {'PASS' if ok else 'FAIL'}  Who we are/Salary 제외, Requirements만 포함")
if not ok:
    print(f"        기대: {expected} / 실제: {result}")

print()
print("레이어 3: 회사 자기언급 필터")
for label, desc, company, expected in layer3_cases:
    test_df = pd.DataFrame({"description": [desc], "company": [company]})
    result = set(tag_dataframe(test_df).loc[0, "skills"].split(", ")) - {""}
    ok = result == expected
    failed += not ok
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
    if not ok:
        print(f"        기대: {expected} / 실제: {result}")

print()
print("레이어 4: 스킬 별칭 병합 (GPT/OpenAI)")
for label, desc, company, expected in layer4_cases:
    test_df = pd.DataFrame({"description": [desc], "company": [company]})
    result = set(tag_dataframe(test_df).loc[0, "skills"].split(", ")) - {""}
    ok = result == expected
    failed += not ok
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
    if not ok:
        print(f"        기대: {expected} / 실제: {result}")

print()
total = len(layer1_cases) + 1 + len(layer3_cases) + len(layer4_cases)
print(f"{total}개 중 {total - failed}개 통과, {failed}개 실패")
assert failed == 0, f"{failed}개 회귀 테스트 실패 — 위 로그에서 어느 케이스인지 확인할 것"
